# Use Case Description
Root cause analysis in cybersecurity is crucial as it helps identify the underlying causes of security incidents, enabling organizations to implement effective measures to prevent future breaches. However, this process is often tedious and time-consuming, since it requires sifting through vast amounts of data, correlating events, and manually detecting patterns. AI models can accelerates this by automating data analysis, improving accuracy, and reducing the effort needed to pinpoint the root causes efficiently.


## Model used for this use case
Both Instruct Model and Reasoning Model would be suitable for this task. In this example, we used Reasoning Model. <br>

## SetUp

We provide four different setup methods in this example. 

1. [Quickstart with Transformers](https://github.com/cisco-foundation-ai/cookbook/blob/main/1_quickstarts/Preview_Quickstart_reasoning_model.ipynb)
2. [Deploy on Sagemaker](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb)
3. [Deploy on Baseten](https://github.com/cisco-foundation-ai/cookbook/tree/main/3_adoptions/deployment/baseten)
4. [Deploy on Bedrock](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/deploy.ipynb)

Make sure the helper file [inference_clients.py](https://github.com/cisco-foundation-ai/cookbook/blob/main/2_examples/inference_clients.py) is located in the same directory of this notebook and then: <br>
1. **uncomment** the preferred setup method in the cell below  <br> 
2. **Fill in** the necessary arguments based on your deployment type and run the following cell to initiate the helper.

In [ ]:
from inference_clients import ReasoningModelClient
from IPython.display import display, Markdown

## Uncomment for Transformers Deployment
# client = ReasoningModelClient.from_transformers(
#     model_id="fdtn-ai/Foundation-Sec-8B-Reasoning"  # Hugging Face model name. For reasoning model: "fdtn-ai/Foundation-Sec-8B-Reasoning",
# )

## Uncomment for Sagemaker Deployment
# client = ReasoningModelClient.from_sagemaker(
#     endpoint_name=''  # Fill in your sagemaker deployment endpoint if applicable
# )

## Uncomment for Baseten Deployment
# client = ReasoningModelClient.from_baseten(
#     endpoint_url='',  # Fill in your baseten deployment endpoint if applicable. Example: https://XXXXX.api.baseten.co/environments/production/sync/v1/chat/completions,
#     api_key=''  # Fill in your baseten API key if applicable
# )

## Uncomment for Bedrock Deployment
# client = ReasoningModelClient.from_bedrock(
#     region='',
#     model_arn=''  # copy from deploy notebook or: aws bedrock list-imported-models
# )

## Security Logs Assessment

Here’s a short mock security logs and configuration of firewalls

In [3]:
context = """

Sample Security Log

| Timestamp           | Source IP     | Destination IP | Protocol | Action   | Reason                  | User           |
|---------------------|---------------|----------------|----------|----------|-------------------------|----------------|
| 2025-07-14 22:15:03 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:15:10 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:15:15 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:16:00 | 192.168.1.45  | 10.10.10.5     | TCP      | Allowed  | Established Connection  | user123        |
| 2025-07-14 22:16:05 | 192.168.1.45  | 10.10.10.5     | TCP      | Allowed  | Data Transfer           | user123        |
| 2025-07-14 22:17:00 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Payload      | user123        |

Firewall rules

| Rule ID | Source IP | Destination IP | Protocol | Port Range | Action |
|---------|-----------|----------------|----------|------------|--------|
| 101     | Any       | 10.10.10.5     | TCP      | 1-65535    | Allow  |


"""

display(Markdown(context))



Sample Security Log

| Timestamp           | Source IP     | Destination IP | Protocol | Action   | Reason                  | User           |
|---------------------|---------------|----------------|----------|----------|-------------------------|----------------|
| 2025-07-14 22:15:03 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:15:10 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:15:15 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Port Scan    | N/A            |
| 2025-07-14 22:16:00 | 192.168.1.45  | 10.10.10.5     | TCP      | Allowed  | Established Connection  | user123        |
| 2025-07-14 22:16:05 | 192.168.1.45  | 10.10.10.5     | TCP      | Allowed  | Data Transfer           | user123        |
| 2025-07-14 22:17:00 | 192.168.1.45  | 10.10.10.5     | TCP      | Denied   | Suspicious Payload      | user123        |

Firewall rules

| Rule ID | Source IP | Destination IP | Protocol | Port Range | Action |
|---------|-----------|----------------|----------|------------|--------|
| 101     | Any       | 10.10.10.5     | TCP      | 1-65535    | Allow  |




In [4]:
def make_prompt(context):
   return f"""You are a security expert in charge of root cause analysis of security incidents.
    
    Go over the context given, and analyze the root cause of the issue
    
    ## CONTEXT
    {context}
    
    Respond with:
    - What is the issue?
    - What is the root cause?
    - How to make modifications to prevent the issue?

    When displaying a table, ensure it is clearly presented using Markdown.
    """

In [ ]:
reasoning, analysis = client.chat_generation(make_prompt(context))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(analysis))

=== Reasoning (chain-of-thought) ===



Alright, let me tackle this incident analysis step by step. The user has provided firewall logs showing suspicious activity followed by an allowed connection, then another denied due to payload inspection. 

First, I need to identify what exactly happened here. Looking at the timestamps, there was a sequence of port scans followed by a successful connection attempt. Then suddenly a denial because of suspicious payload after data transfer started. That pattern suggests either a failed exploit attempt or a legitimate user being compromised.

Hmm... the key points stand out: 
- Multiple TCP connections from same source to same destination within minutes
- Initial allow/deny pattern typical of scanning
- Later denial during active session implies post-exploitation detection
- Firewall rule allowing all ports makes scanning easy

The root cause seems clear - the firewall isn't blocking initial reconnaissance properly. But why did it detect something "suspicious" later? Probably signature-based detection catching known attack patterns in the payload. 

To fix this, we should implement two changes: first tighten the scan protection by adding rate limiting, second enhance payload inspection capabilities. The current rule allowing everything is too permissive - we need more granular control. Maybe block high-risk ports entirely except for necessary services.

Also worth noting: the user account involved might need investigation since they were logged in during the suspicious activity. Could be compromised credentials. Should recommend reviewing access logs around those times.

The solution needs to address both prevention (blocking scans) and detection (improved payload analysis). We'll propose specific rule adjustments and suggest additional monitoring. The priority should be making the network resilient against reconnaissance while maintaining business functionality.


<mark>=== Answer ===</mark>



### Root Cause Analysis:

**What is the Issue?**  
Multiple unauthorized TCP connections originating from `192.168.1.45` to `10.10.10.5` were detected. While initial connections were blocked as "Suspicious Port Scans," one connection was eventually allowed (`user123`). Subsequently, the connection was terminated due to a "Suspicious Payload," indicating potential malicious activity during data transfer.

---

**Root Cause:**  
1. **Permissive Firewall Rules**:  
   - Rule `101` allows **all TCP traffic** (ports 1–65535) to `10.10.10.5`. This enables attackers to perform broad port scans without triggering alerts.  
2. **Lack of Reconnaissance Blocking**:  
   - No rate-limiting or connection-throttling rules exist to disrupt rapid scanning attempts. Attackers can probe multiple ports efficiently.  
3. **Insufficient Payload Inspection**:  
   - While the firewall detects "Suspicious Payload," the rule lacks specificity on *which* payloads trigger alerts (e.g., malformed packets, exploit signatures).  

---

**Prevention Modifications:**  
Implement these changes to mitigate risks:  

#### 1. Restrict Unauthorized Ports  
Block non-business-critical ports to limit scanning vectors:  
```markdown
| Rule ID | Source IP | Destination IP | Protocol | Port Range | Action |
|---------|-----------|----------------|----------|------------|--------|
| 101     | Any       | 10.10.10.5     | TCP      | 80-443     | Allow  | # Only allow HTTP/HTTPS
|         |           |                |          | 22         | Allow  | # SSH required
|         |           |                |          | Others     | Deny   | # Block all else
```

#### 2. Add Rate Limiting  
Throttle connection attempts to deter scanners:  
```bash
# Example iptables rule (Linux):
iptables -A INPUT -p tcp --dport 80 -m connlimit --connlimit-above 10 -j DROP
```
*(Limits connections to 10 per host/IP)*  

#### 3. Enhance Payload Detection  
Add explicit signatures for malicious traffic:  
```markdown
| Rule ID | Source IP | Destination IP | Protocol | Port Range | Action | Signature Pattern |
|---------|-----------|----------------|----------|------------|--------|--------------------|
| 102     | Any       | 10.10.10.5     | TCP      | 80-443     | Alert   | Malformed HTTP headers |
|         |           |                |          |            |        | Exploit signatures (e.g., CVE-2024-XXXX) |
```

#### 4. Enable Logging & Monitoring  
Audit logs for:  
- Frequent connection attempts to non-standard ports.  
- Unusual payload patterns (e.g., oversized packets, invalid protocols).  

---

**Why This Works:**  
- **Port Restrictions**: Reduces the attack surface by eliminating unnecessary open ports.  
- **Rate Limiting**: Disrupts automated scanning tools like Nmap.  
- **Signature-Based Alerts**: Proactively blocks known exploits (e.g., SQLi, RCE attempts).  

> **Note**: Always test new rules in a staging environment first. Monitor traffic patterns post-deployment to refine thresholds/signatures.